# 5. BUILD FACT_SALES

In [0]:
from pyspark.sql.functions import col

df_silver = spark.read.table('retail_project_dev.silver.sales_clean')

fact_sales = df_silver.select('TransactionID', 'CustomerID', 'StoreID','ProductID', col("TranscationDate").cast('date').alias("DateKey"),"DeviceUsed", "PaymentType", "ReturnStatus",
    "Quantity", "Amount", "Discount")

fact_sales.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("retail_project_dev.gold.fact_sales")

# Which product brands are driving the most revenue, and which store regions are performing the best?

In [0]:
%sql
SELECT 
    p.Brand,
    s.StoreRegion,
    SUM(f.Amount) AS TotalSales,
    SUM(f.Quantity) AS UnitsSold
FROM retail_project_dev.gold.fact_sales f
INNER JOIN retail_project_dev.gold.dim_product p ON f.ProductID = p.ProductID
INNER JOIN retail_project_dev.gold.dim_store s ON f.StoreID = s.StoreID
GROUP BY p.Brand, s.StoreRegion
ORDER BY TotalSales DESC
LIMIT 10;